![iSA logo](../iSA_logo.png)
### Materiały do zajęć Statystyka
#### Kurs Junior Data Scientist

**Wojciech Artichowicz**

Przeprowadzono testy A/B pewnej strony internetowej. Sprawdź hipotezę,że zmiany wprowadzone na stronie pozytywnie wpływają na konwersję (czyli ją zwiększają).

In [1]:
import scipy.stats as st
import numpy as np
import pandas as pd

Wczytanie danych

In [2]:
df = pd.read_csv("ab_data.csv") # import danych do obiektu DataFrame
print(df.columns) # wypisanie nazw kolumn
del df[df.columns[0]] # usunięcie kolumny user_id
del df[df.columns[0]] # usunięcie kolumny timestamp
del df[df.columns[0]] # usunięcie kolumny group
df.head() # wypisanie kilku początkowych wierszy danych

Index(['user_id', 'timestamp', 'group', 'landing_page', 'converted'], dtype='object')


,landing_page,converted
0,old_page,0
1,old_page,0
2,new_page,0
3,new_page,0
4,old_page,1


**Wykonanie obliczeń - tabela częstości**

In [3]:
T = pd.crosstab(df["converted"],df["landing_page"])
T

landing_page,new_page,old_page
converted,,
0,129741,129500
1,17498,17739


**Obliczenia związane z testem**

In [4]:
# old_page
nx = sum(T["old_page"])
kx = T["old_page"][1]

# new_page
ny = sum(T["new_page"])
ky = T["new_page"][1]

a = 0.05 # z góry założony poziom istotności

# obliczenie statystyki testowej
u = (kx/nx-ky/ny)/np.sqrt((kx+ky)/(nx*ny)*(1-(kx+ky)/(nx+ny)))

rozkladNormalny = st.norm() # utworzenie instancji rozkładu normalnego
pvalue =  rozkladNormalny.cdf(u) # hipoteza alternatywna lewostronna więc p-value liczy się od lewej strony
                                  # jest to po prostu wartość prawdopodobieństwa dla otrzymanej statystyki testowej
print("p-value={} jest {} niż poziom istotności {}.".format(pvalue,"większa" if pvalue>a else "mniejsza",a))
if pvalue>a:
    print("Brak podstaw do odrzucenia hipotezy zerowej: proporcje są równe.")
else:
    print("Hipotezę zerową należy odrzucić na rzecz hipotezy alternatywnej: proporcja px jest mniejsza od py")

p-value=0.9143962454534289 jest większa niż poziom istotności 0.05.
Brak podstaw do odrzucenia hipotezy zerowej: proporcje są równe.


In [5]:
def statistic(x, y, axis=None):
    return np.sum(x)/len(x) - np.sum(y)/len(y)

x = df.loc[df["landing_page"] == "old_page","converted"].to_numpy()
y = df.loc[df["landing_page"] == "new_page","converted"].to_numpy()

st.permutation_test((x,y),statistic,alternative='less',n_resamples=250).pvalue

0.9282868525896414